In [4]:
"""
Data Pipeline Assignment
========================
Scrape -> Clean -> Convert -> Store (normalized SQLite) -> Query (SQL + pandas)

Data source: https://books.toscrape.com  (public scraping-practice site)

Run:
    pip install requests beautifulsoup4 pandas
    python data_pipeline.py

Output:
    books.db  (SQLite database with `categories` and `books` tables)
    Console output showing scraping progress, cleaning notes,
    all 5 SQL queries with their results, and the pd.read_sql vs
    pd.merge equivalence check.
"""

import re
import sqlite3
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from pathlib import Path





BASE_URL = "https://books.toscrape.com/"
HEADERS = {"User-Agent": "Mozilla/5.0 (data-pipeline-assignment)"}

# Project-defined, fixed baseline conversion rate (not a live/market rate).
GBP_TO_INR = 105.50

MIN_BOOKS = 60
MIN_CATEGORIES = 3

STAR_WORDS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}


# --------------------------------------------------------------------------
# 1. SCRAPING
# --------------------------------------------------------------------------

def get_soup(url: str) -> BeautifulSoup:
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def discover_categories(base_url: str = BASE_URL) -> dict:
    """Return {category_name: category_url} for every category on the site."""
    soup = get_soup(base_url)
    nav_links = soup.select("div.side_categories ul li ul li a")
    categories = {}
    for a in nav_links:
        name = a.get_text(strip=True)
        url = urljoin(base_url, a["href"])
        categories[name] = url
    return categories


def scrape_category(name: str, url: str) -> list:
    """Scrape every book across all paginated pages of one category."""
    books = []
    next_url = url
    while next_url:
        soup = get_soup(next_url)
        for article in soup.select("article.product_pod"):
            title = article.h3.a["title"]
            price_text = article.select_one("p.price_color").get_text(strip=True)
            rating_text = article.select_one("p.star-rating")["class"][1]  # e.g. "Three"
            availability_text = article.select_one(
                "p.instock.availability"
            ).get_text(strip=True)

            books.append(
                {
                    "title": title,
                    "price": price_text,
                    "star_rating": rating_text,
                    "availability": availability_text,
                    "category": name,
                }
            )

        next_link = soup.select_one("li.next a")
        next_url = urljoin(next_url, next_link["href"]) if next_link else None

    return books


def scrape_books(min_books: int = MIN_BOOKS, min_categories: int = MIN_CATEGORIES) -> pd.DataFrame:
    """Scrape at least `min_books` books across at least `min_categories` categories."""
    all_categories = discover_categories()
    all_categories.pop("Books", None)  # umbrella category, mirrors "All products"

    records = []
    used_categories = 0
    for name, url in all_categories.items():
        records.extend(scrape_category(name, url))
        used_categories += 1
        print(f"  scraped category '{name}' -> running total {len(records)} books")
        if len(records) >= min_books and used_categories >= min_categories:
            break

    df = pd.DataFrame(records)
    print(f"Scraped {len(df)} books across {df['category'].nunique()} categories.\n")
    return df


# --------------------------------------------------------------------------
# 2. CLEANING
# --------------------------------------------------------------------------

def parse_price(value):
    try:
        return float(re.sub(r"[^\d.]", "", value))
    except (ValueError, TypeError):
        return None


def parse_rating(value):
    return STAR_WORDS.get(value)  # None if not recognized


def parse_availability(value):
    if not isinstance(value, str):
        return None
    text = value.lower()
    if "in stock" in text:
        return True
    if "out of stock" in text:
        return False
    return None  # unrecognized text


def clean_books(raw_df: pd.DataFrame) -> pd.DataFrame:
    df = raw_df.copy()

    df["price_gbp"] = df["price"].apply(parse_price)
    df["rating"] = df["star_rating"].apply(parse_rating)
    df["in_stock"] = df["availability"].apply(parse_availability)

    # price_gbp is numeric -> median-impute any row that failed to parse
    n_missing_price = df["price_gbp"].isna().sum()
    if n_missing_price:
        median_price = df["price_gbp"].median()
        df["price_gbp"] = df["price_gbp"].fillna(median_price)
        print(f"Median-imputed {n_missing_price} missing price_gbp value(s) "
              f"with {median_price:.2f}.")

    # rating is numeric -> median-impute, then round to a valid 1-5 int
    n_missing_rating = df["rating"].isna().sum()
    if n_missing_rating:
        median_rating = int(round(df["rating"].median()))
        df["rating"] = df["rating"].fillna(median_rating)
        print(f"Median-imputed {n_missing_rating} missing rating value(s) "
              f"with {median_rating}.")
    df["rating"] = df["rating"].astype(int)

    # in_stock is boolean, not numeric -> median-imputation doesn't apply
    # meaningfully to a 2-valued field, so rows with unparseable
    # availability text are dropped instead (documented choice).
    n_missing_stock = df["in_stock"].isna().sum()
    if n_missing_stock:
        print(f"Dropping {n_missing_stock} row(s) with unparseable "
              f"availability text (boolean field -> drop, not impute).")
        df = df.dropna(subset=["in_stock"])
    df["in_stock"] = df["in_stock"].astype(int)  # store as 0/1 for SQLite

    df = df[["title", "category", "price_gbp", "rating", "in_stock"]].reset_index(drop=True)
    return df


def add_inr_price(df: pd.DataFrame, rate: float = GBP_TO_INR) -> pd.DataFrame:
    df = df.copy()
    df["price_inr"] = (df["price_gbp"] * rate).round(2)
    return df


# --------------------------------------------------------------------------
# 3. DATABASE (normalized schema: categories 1---* books)
# --------------------------------------------------------------------------

def build_database(df: pd.DataFrame, db_path: str = "books.db") -> sqlite3.Connection:
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    cur.executescript(
        """
        DROP TABLE IF EXISTS books;
        DROP TABLE IF EXISTS categories;

        CREATE TABLE categories (
            category_id   INTEGER PRIMARY KEY AUTOINCREMENT,
            category_name TEXT UNIQUE NOT NULL
        );

        CREATE TABLE books (
            book_id     INTEGER PRIMARY KEY AUTOINCREMENT,
            title       TEXT NOT NULL,
            price_gbp   REAL NOT NULL,
            price_inr   REAL NOT NULL,
            rating      INTEGER NOT NULL,
            in_stock    INTEGER NOT NULL,
            category_id INTEGER NOT NULL,
            FOREIGN KEY (category_id) REFERENCES categories(category_id)
        );
        """
    )
    conn.commit()

    categories = sorted(df["category"].unique())
    cur.executemany(
        "INSERT INTO categories (category_name) VALUES (?)",
        [(c,) for c in categories],
    )
    conn.commit()

    cat_id_map = dict(
        cur.execute("SELECT category_name, category_id FROM categories").fetchall()
    )

    rows = [
        (
            r.title,
            r.price_gbp,
            r.price_inr,
            r.rating,
            r.in_stock,
            cat_id_map[r.category],
        )
        for r in df.itertuples(index=False)
    ]
    cur.executemany(
        """INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
           VALUES (?, ?, ?, ?, ?, ?)""",
        rows,
    )
    conn.commit()
    print(f"Loaded {len(rows)} books into {db_path} ({len(categories)} categories).\n")
    return conn


# --------------------------------------------------------------------------
# 4. SQL QUERIES (>=5, covering SELECT/WHERE, ORDER BY, LIMIT, DISTINCT,
#    IN/BETWEEN, plus at least one JOIN)
# --------------------------------------------------------------------------

def run_queries(conn: sqlite3.Connection) -> dict:
    queries = {
        "in_stock_expensive": """
            SELECT title, price_gbp, rating
            FROM books
            WHERE in_stock = 1 AND price_gbp > 30
            ORDER BY price_gbp DESC
            LIMIT 10;
        """,
        "distinct_categories": """
            SELECT DISTINCT category_name
            FROM categories
            ORDER BY category_name;
        """,
        "mid_price_books": """
            SELECT title, price_gbp
            FROM books
            WHERE price_gbp BETWEEN 20 AND 40
            ORDER BY price_gbp ASC;
        """,
        "top_rated_books": """
            SELECT title, rating, category_id
            FROM books
            WHERE rating IN (4, 5)
            ORDER BY rating DESC, title ASC
            LIMIT 15;
        """,
        "top_rated_per_category_join": """
            SELECT c.category_name, b.title, b.rating, b.price_gbp, b.price_inr
            FROM books b
            JOIN categories c ON b.category_id = c.category_id
            ORDER BY c.category_name ASC, b.rating DESC
            LIMIT 20;
        """,
    }

    results = {}
    for name, sql in queries.items():
        df_result = pd.read_sql(sql, conn)
        results[name] = {"sql": sql.strip(), "result": df_result}
        print(f"=== {name} ===")
        print(sql.strip())
        print(df_result.to_string(index=False))
        print()

    return results


# --------------------------------------------------------------------------
# 5. pd.read_sql vs pd.merge equivalence check (no SQL for the merge side)
# --------------------------------------------------------------------------

def verify_merge_matches_join(conn: sqlite3.Connection) -> bool:
    books_df = pd.read_sql("SELECT * FROM books;", conn)
    categories_df = pd.read_sql("SELECT * FROM categories;", conn)

    sql_join = """
        SELECT c.category_name, b.title, b.rating, b.price_gbp, b.price_inr
        FROM books b
        JOIN categories c ON b.category_id = c.category_id
        ORDER BY c.category_name ASC, b.rating DESC
        LIMIT 20;
    """
    sql_result = pd.read_sql(sql_join, conn).reset_index(drop=True)

    merged = (
        books_df.merge(categories_df, on="category_id", how="inner")[
            ["category_name", "title", "rating", "price_gbp", "price_inr"]
        ]
        .sort_values(["category_name", "rating"], ascending=[True, False])
        .head(20)
        .reset_index(drop=True)
    )

    print("--- pd.read_sql (JOIN query) result ---")
    print(sql_result.to_string(index=False))
    print("\n--- pd.merge (in-memory, no SQL) result ---")
    print(merged.to_string(index=False))

    match = sql_result.equals(merged)
    print(f"\nSQL JOIN and pandas merge results match: {match}")
    return match


# --------------------------------------------------------------------------
# MAIN
# --------------------------------------------------------------------------

def main():
    print(f"Fixed baseline conversion rate: 1 GBP = {GBP_TO_INR} INR\n")

    print("Scraping...")
    raw_df = scrape_books()

    print("Cleaning...")
    clean_df = clean_books(raw_df)

    print("Converting currency...")
    priced_df = add_inr_price(clean_df)

    print("Loading into SQLite...")
    conn = build_database(priced_df, db_path="books.db")

    print("Running SQL queries...\n")
    run_queries(conn)

    print("Verifying pd.merge against SQL JOIN...\n")
    verify_merge_matches_join(conn)

    conn.close()
    print("\nDone. Database saved to books.db")
print("PK")
print("FK")

if __name__ == "__main__":
    main()

PK
FK
Fixed baseline conversion rate: 1 GBP = 105.5 INR

Scraping...
  scraped category 'Travel' -> running total 11 books
  scraped category 'Mystery' -> running total 43 books
  scraped category 'Historical Fiction' -> running total 69 books
Scraped 69 books across 3 categories.

Cleaning...
Converting currency...
Loading into SQLite...
Loaded 69 books into books.db (3 categories).

Running SQL queries...

=== in_stock_expensive ===
SELECT title, price_gbp, rating
            FROM books
            WHERE in_stock = 1 AND price_gbp > 30
            ORDER BY price_gbp DESC
            LIMIT 10;
                                                                 title  price_gbp  rating
                                         Boar Island (Anna Pigeon #19)      59.48       3
The No. 1 Ladies' Detective Agency (No. 1 Ladies' Detective Agency #1)      57.70       4
                                      A Year in Provence (Provence #1)      56.88       4
                                      